<a href="https://colab.research.google.com/github/marsya505/DataMining/blob/main/Week13-Latihan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import (StandardScaler, LabelEncoder, OneHotEncoder)
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix)
from sklearn.neural_network import MLPClassifier

In [ ]:
# TITANIC — BACKPROPAGATION
np.random.seed(42)
n = 891
survived = np.random.choice([0, 1], n, p=[0.38, 0.62])
pclass   = np.random.choice([1, 2, 3], n, p=[0.24, 0.21, 0.55])
sex      = np.random.choice([0, 1], n, p=[0.65, 0.35])   # 0=male, 1=female
age      = np.clip(np.random.normal(29.7, 14.5, n), 1, 80)
sibsp    = np.random.choice([0,1,2,3,4,5], n, p=[0.68,0.23,0.05,0.02,0.01,0.01])
parch    = np.random.choice([0,1,2,3,4,5], n, p=[0.76,0.13,0.09,0.01,0.005,0.005])
fare     = np.clip(np.random.exponential(32, n), 5, 512)
embarked = np.random.choice([0, 1, 2], n, p=[0.72, 0.19, 0.09])

df_titanic = pd.DataFrame({
    'Survived': survived, 'Pclass': pclass, 'Sex': sex,
    'Age': age, 'SibSp': sibsp, 'Parch': parch,
    'Fare': fare, 'Embarked': embarked
})

print("Titanic Dataset")
print(f"\nJumlah data : {df_titanic.shape[0]} baris, {df_titanic.shape[1]} kolom")
print(f"Fitur       : Pclass, Sex, Age, SibSp, Parch, Fare, Embarked")
print(f"Target      : Survived  (0=Tidak Selamat | 1=Selamat)")
print(f"\nDistribusi Survived:\n{df_titanic['Survived'].value_counts().to_string()}")

# Preprocessing
features  = ['Pclass','Sex','Age','SibSp','Parch','Fare','Embarked']
X_titanic = df_titanic[features].values
y_titanic = df_titanic['Survived'].values

encoder  = OneHotEncoder(sparse_output=False)
y_onehot = encoder.fit_transform(y_titanic.reshape(-1, 1))

X_tr, X_te, y_tr, y_te = train_test_split(
    X_titanic, y_onehot, test_size=0.2, random_state=42
)
y_te_labels = np.argmax(y_te, axis=1)

scaler = StandardScaler()
X_tr   = scaler.fit_transform(X_tr)
X_te   = scaler.transform(X_te)

# Fungsi aktivasi Sigmoid
def sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

def sigmoid_derivative(x):
    return x * (1 - x)

# Parameter jaringan
input_neurons  = X_tr.shape[1]   # 7
hidden_neurons = 10
output_neurons = y_tr.shape[1]   # 2

np.random.seed(42)
W1 = np.random.uniform(-0.5, 0.5, (input_neurons, hidden_neurons))
W2 = np.random.uniform(-0.5, 0.5, (hidden_neurons, output_neurons))
b1 = np.zeros((1, hidden_neurons))
b2 = np.zeros((1, output_neurons))

learning_rate = 0.01
epochs        = 5000
loss_history  = []

print(f"\nArsitektur  : {input_neurons} → {hidden_neurons} → {output_neurons}")
print(f"Learning Rate: {learning_rate} | Epochs: {epochs}\n")

# Training Loop
for epoch in range(epochs):
    # Forward propagation
    z1     = np.dot(X_tr, W1) + b1
    a1     = sigmoid(z1)
    z2     = np.dot(a1, W2) + b2
    output = sigmoid(z2)

    error  = y_tr - output
    loss   = np.mean(np.square(error))
    loss_history.append(loss)

    # Backpropagation
    d_output = error * sigmoid_derivative(output)
    error_h  = d_output.dot(W2.T)
    d_hidden = error_h * sigmoid_derivative(a1)

    # Update bobot & bias
    W2 += a1.T.dot(d_output) * learning_rate
    b2 += np.sum(d_output, axis=0, keepdims=True) * learning_rate
    W1 += X_tr.T.dot(d_hidden) * learning_rate
    b1 += np.sum(d_hidden, axis=0, keepdims=True) * learning_rate

    if (epoch + 1) % 1000 == 0:
        print(f"  Epoch {epoch+1:5d} | Loss: {loss:.6f}")

# Evaluasi
a1_test     = sigmoid(np.dot(X_te, W1) + b1)
out_test    = sigmoid(np.dot(a1_test, W2) + b2)
pred_labels = np.argmax(out_test, axis=1)
acc_bp      = accuracy_score(y_te_labels, pred_labels)

print(f"\nAkurasi Backpropagation (Test) : {acc_bp * 100:.2f}%")
print("\nConfusion Matrix:")
print(confusion_matrix(y_te_labels, pred_labels))
print("\nClassification Report:")
print(classification_report(y_te_labels, pred_labels, target_names=['Tidak Selamat', 'Selamat']))

In [ ]:

# MUSHROOM CLASSIFICATION

# Preprocessing — Label Encoding semua fitur kategorikal
df_enc = df_m.copy()
le = LabelEncoder()
for col in df_enc.columns:
    df_enc[col] = le.fit_transform(df_enc[col])

X_m = df_enc.drop('class', axis=1).values
y_m = df_enc['class'].values

X_m_tr, X_m_te, y_m_tr, y_m_te = train_test_split(
    X_m, y_m, test_size=0.2, random_state=42
)
sc2   = StandardScaler()
X_m_tr = sc2.fit_transform(X_m_tr)
X_m_te  = sc2.transform(X_m_te)

print("Mushroom Classification Dataset")
print(f"\nData training : {X_m_tr.shape[0]} sampel")
print(f"Data testing  : {X_m_te.shape[0]} sampel")

# Eksperimen Hidden Layer Size
print("\nHIDDEN LAYER SIZE (activation=relu)\n")

hidden_layers = [
    (8, 8, 8),
    (10, 10, 6),
    (6, 8, 9),
    (6, 9, 1),
    (6, 3, 9),
    (7, 10, 8),
    (16, 8),
    (32, 16, 8),
]
results_hl = []

for hl in hidden_layers:
    mlp = MLPClassifier(hidden_layer_sizes=hl, activation='relu', solver='adam', max_iter=500, random_state=42)
    mlp.fit(X_m_tr, y_m_tr)
    pred_tr = mlp.predict(X_m_tr)
    pred_te = mlp.predict(X_m_te)
    a_tr = accuracy_score(y_m_tr, pred_tr)
    a_te = accuracy_score(y_m_te, pred_te)
    results_hl.append({'Hidden Layer': str(hl), 'Acc Train': a_tr, 'Acc Test': a_te})
    print(f"  Hidden Layer {str(hl):15s} | "f"Train: {a_tr*100:.2f}%  Test: {a_te*100:.2f}%")
    print(f"    Confusion Matrix: {confusion_matrix(y_m_te, pred_te).tolist()}")

df_hl = pd.DataFrame(results_hl).sort_values('Acc Test', ascending=False)
print("\nHIDDEN LAYER (urut test accuracy):")
print(df_hl.to_string(index=False))
best_hl     = df_hl.iloc[0]
best_hl_tup = eval(best_hl['Hidden Layer'])
print(f"\nTERBAIK: {best_hl['Hidden Layer']}  " f"Test Acc: {best_hl['Acc Test']*100:.2f}%")

# Activation Function
activations = ['relu', 'tanh', 'logistic', 'identity']
results_act = []

for act in activations:
    mlp = MLPClassifier(hidden_layer_sizes=best_hl_tup, activation=act,
                        solver='adam', max_iter=500, random_state=42)
    mlp.fit(X_m_tr, y_m_tr)
    pred_tr = mlp.predict(X_m_tr)
    pred_te = mlp.predict(X_m_te)
    a_tr = accuracy_score(y_m_tr, pred_tr)
    a_te = accuracy_score(y_m_te, pred_te)
    results_act.append({'Activation': act, 'Acc Train': a_tr, 'Acc Test': a_te})
    print(f"  Activation: {act}")
    print(f"    Train: {a_tr*100:.2f}%  |  Test: {a_te*100:.2f}%")
    print(f"    Classification Report (Test):")
    print(classification_report(y_m_te, pred_te, target_names=['Edible','Poisonous'], zero_division=0))

df_act = pd.DataFrame(results_act).sort_values('Acc Test', ascending=False)
print("\nACTIVATION (urut test accuracy):")
print(df_act.to_string(index=False))
best_act = df_act.iloc[0]
print(f"\nTERBAIK: {best_act['Activation']}  " f"Test Acc: {best_act['Acc Test']*100:.2f}%")

print("KESIMPULAN")
print(f"  Dataset          : Mushroom Classification UCI")
print(f"  Jumlah Sampel    : 8.124 jamur (22 fitur kategorikal)")
print(f"  Best Hidden Layer: {best_hl['Hidden Layer']}")
print(f"  Best Activation  : {best_act['Activation']}")
print(f"  Best Test Acc    : {best_act['Acc Test']*100:.2f}%")
print(f"\n  Kombinasi terbaik:")
print(f"    MLPClassifier(")
print(f"      hidden_layer_sizes = {best_hl_tup},")
print(f"      activation         = '{best_act['Activation']}',")
print(f"      solver             = 'adam',")
print(f"      max_iter           = 500)")

# VISUALISASI
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Hasil Eksperimen ANN — Titanic & Mushroom Classification', fontsize=13, fontweight='bold')

# Plot 1: Loss Curve Titanic
axes[0].plot(range(1, len(loss_history)+1), loss_history, color='steelblue', linewidth=1.5)
axes[0].set_title('Titanic — Training Loss\n(Backpropagation Manual)')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')
axes[0].grid(alpha=0.3)
axes[0].set_yscale('log')

# Plot 2: Hidden Layer (Mushroom)
df_hl_p = pd.DataFrame(results_hl)
x  = np.arange(len(df_hl_p))
w  = 0.35
b1_bar = axes[1].bar(x - w/2, df_hl_p['Acc Train']*100, w,
                     label='Train', color='steelblue', alpha=0.85)
b2_bar = axes[1].bar(x + w/2, df_hl_p['Acc Test']*100,  w,
                     label='Test',  color='coral', alpha=0.85)
best_i = df_hl_p['Acc Test'].idxmax()
b2_bar[best_i].set_edgecolor('green')
b2_bar[best_i].set_linewidth(2.5)
axes[1].set_title('Mushroom — Hidden Layer Size')
axes[1].set_xlabel('Hidden Layer')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_xticks(x)
axes[1].set_xticklabels(df_hl_p['Hidden Layer'],
                         rotation=30, ha='right', fontsize=7)
axes[1].set_ylim(40, 90)
axes[1].legend(); axes[1].grid(axis='y', alpha=0.3)
for bar in list(b1_bar) + list(b2_bar):
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.3,
                 f'{bar.get_height():.1f}%',
                 ha='center', va='bottom', fontsize=6)

# Plot 3: Activation (Mushroom)
df_act_p = pd.DataFrame(results_act)
x3 = np.arange(len(df_act_p))
b3_bar = axes[2].bar(x3 - w/2, df_act_p['Acc Train']*100, w,
                     label='Train', color='steelblue', alpha=0.85)
b4_bar = axes[2].bar(x3 + w/2, df_act_p['Acc Test']*100,  w,
                     label='Test',  color='coral', alpha=0.85)
best_j = df_act_p['Acc Test'].idxmax()
b4_bar[best_j].set_edgecolor('green')
b4_bar[best_j].set_linewidth(2.5)
axes[2].set_title('Mushroom — Activation Function')
axes[2].set_xlabel('Activation')
axes[2].set_ylabel('Accuracy (%)')
axes[2].set_xticks(x3)
axes[2].set_xticklabels(df_act_p['Activation'], fontsize=10)
axes[2].set_ylim(40, 90)
axes[2].legend(); axes[2].grid(axis='y', alpha=0.3)
for bar in list(b3_bar) + list(b4_bar):
    axes[2].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.3,
                 f'{bar.get_height():.1f}%',
                 ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig('hasil_ann_titanic_mushroom.png', dpi=150, bbox_inches='tight')
plt.show()